# Lab results import: analysis

Two frames, both read live from the database every time you call them, so
there is nothing to invalidate. Neither writes anything.

- `get_df_orphan_results()` — imported results carrying no requisition,
  bucketed by what has to happen to each.
- `get_df_result_comparison()` — every result CRF value against the
  imported result. One row per CRF instance per utest id, blanks kept, so
  it is a complete grid of what could have been keyed rather than only
  what was.

See `../README.rst` for the runbooks and for the commands that do write.

In [ ]:
# Run under `manage.py shell_plus --notebook`, or configure Django first.
#
# Import Result explicitly. `edc_lab.models.Result` is a different model
# with a `panel` FK and no `panel_name`, and shell_plus will happily
# shadow one with the other, which makes every query quietly wrong.
import pandas as pd

from edc_lab_results_import.dataframes import (
    changed_since_pulled,
    get_df_orphan_results,
    get_df_result_comparison,
)
from edc_lab_results_import.models import Result

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

---
## 1. Orphans: imported results with no requisition

In [ ]:
df_orphans = get_df_orphan_results()

n_orphans = Result.objects.filter(requisition__isnull=True).count()
print(f"{n_orphans:>8}  orphaned results (model)")
print(f"{len(df_orphans):>8}  rows in the frame")
df_orphans.bucket.value_counts()

What each bucket means, most actionable first.

| bucket | meaning | action |
| --- | --- | --- |
| `resolver_miss` | a requisition already exists at this timepoint for this panel | `manage.py link_orphan_results` |
| `requisition_not_keyed` | the panel was expected here and nobody keyed it | data manager |
| `panel_unknown` | the utest id is in no registered panel | fix the mapping, see `get_mappings` |
| `panel_not_expected` | a real panel at a timepoint that did not call for it | ad hoc draw, or review the mapping |
| `visit_not_found` | no timepoint | see section 1.2 |

The buckets are ordered and a row can be in more than one state, so read
them as "what to do first", not as disjoint causes.

In [ ]:
# A data manager keys a requisition, not an analyte. This is the real
# size of the work, not the row count.
df_orphans.groupby(
    ["subject_identifier", "visit_code", "visit_code_sequence", "panel_name"]
).ngroups

### 1.1 Health checks

Both of these should be empty on a correctly configured system. Anything
here is a configuration problem, not a data problem.

In [ ]:
# Utest ids in no registered panel. Non-empty means the utest id to panel
# mapping is incomplete, and those results can never reach a requisition.
df_orphans.loc[df_orphans.panel_name.isna(), "utestid"].value_counts().head(20)

In [ ]:
# Analyte panel vs the panel it was drawn under. A large `panel_not_expected`
# concentrated on one panel usually means
# EDC_LAB_RESULTS_IMPORT_REQUISITION_PANEL_MAP is missing an entry.
(
    df_orphans.loc[df_orphans.bucket == "panel_not_expected"]
    .groupby(["panel_name", "requisition_panel_name"])
    .size()
    .sort_values(ascending=False)
    .head(10)
)

### 1.2 Results with no timepoint

Split three ways: no subject at all (identity resolution, nothing here can
help), drawn on or before the subject's first visit (the baseline rule can
place them), and drawn after baseline (still unsolved).

In [ ]:
vnf = df_orphans.loc[df_orphans.bucket == "visit_not_found"]

print(f"{len(vnf):>8}  no timepoint")
print(f"{vnf.subject_identifier.isna().sum():>8}  ... and no subject identifier either")
print(f"{vnf.candidate_rule.notna().sum():>8}  ... proposed for baseline")
n_waiting = vnf.candidate_requisition_id.notna().sum()
print(f"{n_waiting:>8}  ... and a requisition already waiting")

In [ ]:
# How far before baseline were they drawn? Bounded by
# MAX_DAYS_BEFORE_BASELINE, so this is the shape inside that window.
df_orphans.loc[df_orphans.candidate_rule.notna(), "days_before_baseline"].describe()

---
## 2. Comparison: imported values against the CRF

In [ ]:
df = get_df_result_comparison()

print(f"{len(df):>8}  grid rows (result CRF instances x utest ids)")
df.value_status.value_counts()

In [ ]:
df.units_status.value_counts()

`units_status` is `not_compared` **only** where nothing was imported for
the row, so that count is exactly the no-import population: CRF grid rows
with nothing from the lab to check against.

`value_status` of `not_compared` is that same population plus the rows
whose units differ from the CRF, where the value cannot be expressed in
the CRF's units and so is never compared. The difference between the two
counts is that second group.

In [ ]:
no_import = (df.units_status == "not_compared").sum()
units_blocked = (df.value_status == "not_compared").sum() - no_import

print(f"{no_import:>8}  CRF rows with no imported result")
print(f"{units_blocked:>8}  imported, but units differ so the value was never compared")

### 2.1 Split `differs` before working it

`value_status` calls one blank side a difference. On the result search page
that is right, since rows there are built from imported results against an
existing CRF. In this grid it is not: a blank CRF value with a lab result
waiting has not been transcribed yet, which is a different job from a value
transcribed wrongly.

In [ ]:
differs = df.loc[df.value_status == "differs"]
both = differs.loc[differs.crf_value.notna() & differs.result_value.notna()]
untranscribed = differs.loc[differs.crf_value.isna()]

print(f"{len(differs):>8}  differs")
print(f"{len(both):>8}  ... both values present, a real disagreement")
print(f"{len(untranscribed):>8}  ... CRF blank, not yet transcribed")
print(f"{differs.result_value.isna().sum():>8}  ... CRF present, no lab value")

### 2.2 Is it precision, or is it error?

The lab usually reports more digits than the CRF stores. A CRF field at
`decimal_places=2` holding 13.40 against an imported 13.4400 is a genuine
difference at the stored precision, and it is flagged, but nobody typed
anything wrong.

A median `pct_diff` well under 1% means you are looking at precision, not
transcription. The workload is in the tail.

In [ ]:
both.pct_diff.describe()

In [ ]:
# The workload at each threshold. Pick one, there is no correct answer.
for threshold in [0.5, 1, 2, 5, 10, 25, 50]:
    print(f"  >= {threshold:>5}% : {(both.pct_diff >= threshold).sum():>8}")

### 2.3 Patterns worth looking for

A ratio at a power of ten is a decimal point slip, which a percentage sort
buries among the genuinely large differences. A ratio at some other fixed
factor across one analyte is a unit conversion nobody is doing.

In [ ]:
both.ratio.round(1).value_counts().head(12)

In [ ]:
# Decimal point slips specifically.
slips = both.loc[both.ratio.round(2).isin([0.01, 0.1, 10.0, 100.0])]
print(f"{len(slips)} decimal point slip(s)")
slips[
    [
        "subject_identifier",
        "visit_code",
        "utestid",
        "result_value",
        "crf_value",
        "units",
        "ratio",
    ]
].head(20)

In [ ]:
# Concentrated on a few analytes, or spread? Concentration points at a
# scale or units fault on those assays rather than at data entry.
both.utestid.value_counts().head(20)

In [ ]:
# A corrected report filed against the same requisition produces two rows
# for one CRF value, and at most one can match, so anything above 1 here
# inflates `differs` mechanically.
df.n_imported_for_key.value_counts()

### 2.4 The review worklist

Set the threshold from section 2.2 rather than taking this default on
faith.

In [ ]:
WORKLIST_THRESHOLD = 5.0

worklist = (
    both.loc[both.pct_diff >= WORKLIST_THRESHOLD]
    .sort_values("pct_diff", ascending=False)
    .loc[
        :,
        [
            "subject_identifier",
            "visit_code",
            "panel_name",
            "utestid",
            "result_value",
            "units",
            "crf_value",
            "crf_units",
            "abs_diff",
            "pct_diff",
            "ratio",
            "requisition_identifier",
            "source_file",
        ],
    ]
)
print(f"{len(worklist)} row(s) at >= {WORKLIST_THRESHOLD}%")
worklist.head(30)

### 2.5 Units that never get compared

Nothing computes `converted_result_value` or `converted_units`:
`apply_unit_mapping_after_resolve` only rewrites `units` in place. So a
result whose units differ from the CRF is never converted and never
compared, however close the values are.

Anything frequent here is a candidate for the unit mapping.

In [ ]:
(
    df.loc[df.units_status == "differs"]
    .groupby(["utestid", "units", "crf_units"])
    .size()
    .sort_values(ascending=False)
    .head(20)
)

### 2.6 The requisition contradicts the results

`result_expected` of NO says the lab was never going to report, yet
results arrived against that requisition. Someone should look.

In [ ]:
conflicts = df.loc[df.result_expected_conflict]
print(f"{len(conflicts)} row(s)")
conflicts[
    [
        "subject_identifier",
        "visit_code",
        "panel_name",
        "utestid",
        "result_value",
        "result_not_expected_reason",
        "requisition_identifier",
    ]
].head(20)

---
## 3. Has the frame gone stale?

The frames are queries, so re-running a cell always reflects the database.
What goes stale is a frame held in this notebook while someone edits
requisitions, `result_expected` in particular. Any non-zero count means
read it again.

Note `DataFrame.attrs` survives a notebook but not a round trip through
CSV, so an exported worklist loses its timestamp.

In [ ]:
changed_since_pulled(df)